# Reproducing the paper's SUPPORT numbers

The paper reports on SUPPORT a C-index of 0.84 for Cox and 0.85 for VaDeSC (Table 4, 5 train-test splits). With the same variables (including the post-baseline ones) our setup gives about 0.80 and 0.77-0.78. Here we find out why, using the official preprocessing from the VaDeSC repository (`datasets/support/support_data.py`), copied in `src/paper_setup.py`, and the five seeds of the official run script (`bin/run_vadesc_support`: 42, 73, 666, 777, 1009). Each seed gives a different split.

Differences between the official preprocessing and ours:

| | official | ours |
|---|---|---|
| missing category (e.g. `sfdm2`, `income`) | all one-hot columns 0 | most frequent training category |
| log transform (`totcst`, `charges`, `pafi`, `sod`) | abs(log(x)) | log(1 + x) |
| scaler fitted on | all patients | training set |
| split | random 80 / 10 (test) / 10, not stratified | 70 / 10 / 20 (test), stratified on the event |

Section 1 switches these on one at a time with Cox (fast and deterministic). Section 2 runs VaDeSC in both setups.

In [1]:
import sys
sys.path.append("..")
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd

from src.baselines import run_cox, run_weibull_aft
from src.data import load_support, load_support_raw
from src.paper_setup import load_support_variant
from src.vadesc import evaluate_vadesc, train_vadesc

SEEDS = [42, 73, 666, 777, 1009]  # official run script
CFG = dict(K=4, d_lat=16, weibull_k=2.0, epochs=300)  # configs/support.yml
F = ["missing_cat_as_zeros", "official_log", "scaler_on_all", "official_split"]
summ = lambda r: f"{np.mean(r):.3f} ± {np.std(r, ddof=1):.3f}"

## 1. Which difference matters? (Cox)

In [2]:
rows = {"our loader (load_support)": [run_cox(load_support(seed=s))["C-index"] for s in SEEDS],
        "all differences off": [run_cox(load_support_variant(s, **{f: False for f in F}))["C-index"] for s in SEEDS]}
for f in F:
    rows[f"only {f}"] = [run_cox(load_support_variant(s, **{g: g == f for g in F}))["C-index"] for s in SEEDS]
rows["official (all on)"] = [run_cox(load_support_variant(s))["C-index"] for s in SEEDS]
abl = pd.DataFrame(rows, index=SEEDS).T
abl["mean"], abl["std"] = abl[SEEDS].mean(1), abl[SEEDS].std(1)
abl.round(3)

,42,73,666,777,1009,mean,std
our loader (load_support),0.805,0.803,0.809,0.798,0.792,0.801,0.006
all differences off,0.805,0.803,0.809,0.798,0.792,0.801,0.006
only missing_cat_as_zeros,0.841,0.846,0.848,0.834,0.835,0.841,0.006
only official_log,0.805,0.803,0.809,0.799,0.792,0.801,0.006
only scaler_on_all,0.805,0.803,0.809,0.798,0.792,0.801,0.006
only official_split,0.813,0.806,0.796,0.790,0.807,0.803,0.009
official (all on),0.858,0.845,0.837,0.837,0.843,0.844,0.008


"All differences off" gives exactly the numbers of our own loader, so the variant loader is correct. Almost the whole gap comes from **how a missing category is encoded**. The reason is `sfdm2`:

In [3]:
raw = load_support_raw()
early = (raw["d.time"] <= 60) & (raw["death"] == 1)
print(raw["sfdm2"].value_counts(dropna=False).to_string())
print(f"\nsfdm2 missing for {raw.sfdm2.isna().sum()} patients; died within 2 months: {early[raw.sfdm2.isna()].mean():.0%} of them "
      f"vs {early[raw.sfdm2.notna()].mean():.0%} of the others")
print(f'most frequent level (what our loader fills in): "{raw.sfdm2.mode()[0]}"')

sfdm2
<2 mo. follow-up       3123
no(M2 and SIP pres)    3061
NaN                    1400
adl>=4 (>=5 if sur)     916
SIP>=30                 564
Coma or Intub            41

sfdm2 missing for 1400 patients; died within 2 months: 2% of them vs 41% of the others
most frequent level (what our loader fills in): "<2 mo. follow-up"


- `sfdm2` is missing for 1,400 patients, and 98% of them survived the first two months.
- Our loader fills a missing category with the most frequent one, which for `sfdm2` is `"<2 mo. follow-up"`, the level that means early death. So 1,400 survivors get the "died early" label, which blurs the leak.
- The official encoding (all zeros) keeps `sfdm2` an almost perfect early-death signal, so every model gets a higher C-index.

Without the post-baseline variables the encoding does not matter:

In [4]:
off = dict(official_log=False, scaler_on_all=False, official_split=False)
for name, kw in [("baseline-only, missing category = mode (ours)", dict(missing_cat_as_zeros=False, drop_post_baseline=True, **off)),
                 ("baseline-only, missing category = zeros", dict(missing_cat_as_zeros=True, drop_post_baseline=True, **off)),
                 ("baseline-only, official preprocessing", dict(drop_post_baseline=True))]:
    print(f"{name}: Cox C-index {summ([run_cox(load_support_variant(s, **kw))['C-index'] for s in SEEDS])}")

baseline-only, missing category = mode (ours): Cox C-index 0.732 ± 0.006


baseline-only, missing category = zeros: Cox C-index 0.732 ± 0.006


baseline-only, official preprocessing: Cox C-index 0.728 ± 0.008


## 2. VaDeSC and the baselines in both setups

Same five seeds (seed = split and initialisation). Paper (Table 4): Cox PH 0.84 ± 0.01, Weibull AFT 0.84 ± 0.01, VaDeSC 0.85 ± 0.01.

In [5]:
SETUPS = {
    "official preprocessing (with post-baseline)": lambda s: load_support_variant(s),
    "our preprocessing (with post-baseline)": lambda s: load_support(seed=s),
    "official preprocessing, baseline-only": lambda s: load_support_variant(s, drop_post_baseline=True),
}
rows = []
for name, load in SETUPS.items():
    for s in SEEDS:
        d = load(s)
        rows.append({"setup": name, "seed": s, "model": "Cox PH", **run_cox(d)})
        rows.append({"setup": name, "seed": s, "model": "Weibull AFT", **run_weibull_aft(d)})
        m, _, _ = evaluate_vadesc(train_vadesc(d, seed=s, **CFG), d)
        rows.append({"setup": name, "seed": s, "model": "VaDeSC", "C-index": m["C-index"], "IBS": m["IBS"]})
        print(name, s, "done", flush=True)
res = pd.DataFrame(rows)
res.to_csv("../results/paper_setup_by_seed.csv", index=False)
(res.groupby(["setup", "model"])[["C-index", "IBS"]].agg(["mean", "std"])
    .reindex(list(SETUPS), level=0).reindex(["Cox PH", "Weibull AFT", "VaDeSC"], level=1).round(3))

official preprocessing (with post-baseline) 42 done


official preprocessing (with post-baseline) 73 done


official preprocessing (with post-baseline) 666 done


official preprocessing (with post-baseline) 777 done


official preprocessing (with post-baseline) 1009 done


our preprocessing (with post-baseline) 42 done


... (6 outputs left out)

official preprocessing, baseline-only 666 done


official preprocessing, baseline-only 777 done


official preprocessing, baseline-only 1009 done


C-index           IBS  \
                                                           mean    std   mean   
setup                                       model                               
official preprocessing (with post-baseline) Cox PH        0.844  0.008  0.119   
                                            Weibull AFT   0.844  0.008  0.118   
                                            VaDeSC        0.857  0.007  0.118   
our preprocessing (with post-baseline)      Cox PH        0.801  0.006  0.152   
                                            Weibull AFT   0.802  0.005  0.149   
                                            VaDeSC        0.792  0.009  0.159   
official preprocessing, baseline-only       Cox PH        0.728  0.008  0.178   
                                            Weibull AFT   0.726  0.008  0.180   
                                            VaDeSC        0.671  0.020  0.200   

                                                                
                                                           std  
setup                                       model               
official preprocessing (with post-baseline) Cox PH       0.006  
                                            Weibull AFT  0.006  
                                            VaDeSC       0.008  
our preprocessing (with post-baseline)      Cox PH       0.006  
                                            Weibull AFT  0.004  
                                            VaDeSC       0.003  
official preprocessing, baseline-only       Cox PH       0.004  
                                            Weibull AFT  0.004  
                                            VaDeSC       0.007

## Notes

- **With the official preprocessing we reproduce the paper**: Cox 0.844 ± 0.008 (paper 0.84 ± 0.01), Weibull AFT 0.844 ± 0.008 (paper 0.84), VaDeSC 0.857 ± 0.007 (paper 0.85 ± 0.01). So our PyTorch VaDeSC matches the original, and the lower numbers in our other notebooks come from the data setup, not from the model.
- **The gap is almost entirely the encoding of the missing `sfdm2` values.** 1,400 patients have no 2-month score, and 98% of them survived the first two months. The official one-hot encoding leaves them all zero, so `sfdm2` stays an almost perfect early-death signal. Our loader fills them with the most frequent level, `"<2 mo. follow-up"`, which blurs the leak (Cox 0.80). The other differences (log transform, scaler, split) change the C-index by less than 0.01.
- So the paper's SUPPORT results depend even more on the post-baseline variable than our leakage comparison suggested: with it, all models reach 0.84-0.86; without it (official preprocessing, baseline-only) Cox gets 0.728 and VaDeSC 0.671 ± 0.020, the same as with our preprocessing (notebook 10: 0.657-0.669). Without the leak, the encoding makes no difference (Cox 0.732 both ways).
- VaDeSC is slightly above Cox only when the leak is present (0.857 vs 0.844); without it, it is clearly below (0.671 vs 0.728).
